# OncoBridge C2: referencia 3D MedDiffusion en Google Colab

Este notebook usa la implementación oficial de [ShanghaiTech-IMPACT/3D-MedDiffusion](https://github.com/ShanghaiTech-IMPACT/3D-MedDiffusion) para generar un volumen anatómico CT/MRI y exportar un corte PNG que puede cargarse como referencia en el Componente 2.

> **Límite esencial:** los checkpoints públicos son condicionales por anatomía y modalidad, no por el prompt de C1 ni por patología. El resultado no representa un paciente, no garantiza una lesión y no debe utilizarse como evidencia clínica.

La documentación oficial indica al menos **40 GB de VRAM** para inferencia. Elegí en Colab una GPU A100 de 40 GB o superior; una T4/L4 estándar no está soportada para este flujo.

## 1. Preparar el entorno

En Colab: `Runtime > Change runtime type > GPU`. Confirmá que la GPU tenga memoria suficiente antes de continuar.

In [ ]:
!nvidia-smi
!git clone https://github.com/ShanghaiTech-IMPACT/3D-MedDiffusion.git
%cd 3D-MedDiffusion
!pip install -q torchio nibabel pytorch-lightning monai==1.4.0 monai-generative==0.2.3 einops einops-exts rotary-embedding-torch scikit-image gdown

## 2. Descargar checkpoints oficiales

El repositorio oficial publica los checkpoints en Google Drive. La celda intenta descargar la carpeta; si Drive solicita autorización o la estructura cambia, descargá manualmente los pesos desde el enlace oficial y subilos a `checkpoints/`. Para el flujo 8x se esperan `PatchVolume_8x_s2.ckpt` y `BiFlowNet_0453500.pt`.

In [ ]:
!mkdir -p checkpoints
!gdown --folder 'https://drive.google.com/drive/folders/1h1Ina5iUkjfSAyvM5rUs4n1iqg33zB-J?usp=drive_link' -O checkpoints
!find checkpoints -type f | sort

## 3. Elegir anatomía

Usá la modalidad indicada por el output de C1. Mapeo disponible en los pesos oficiales:

| C1 / región | `CLASS_ID` | `LATENT_RESOLUTION` |
|---|---:|---|
| CT de cabeza/cuello | 0 | `(16, 32, 32)` |
| CT de tórax o abdomen | 1 | `(16, 32, 32)` |
| MRI de abdomen | 5 | `(16, 32, 32)` |

La resolución elegida produce un volumen de 128×256×256 con el modelo 8x. No hay control por lesión, lateralidad, biomarcadores o identidad del paciente.

In [ ]:
import json
from google.colab import files

# Subí aquí el c1_case_001.json creado por OncoBridge.
uploaded = files.upload()
if not uploaded:
    raise ValueError('Subí un output JSON de C1 para seleccionar la anatomía.')
c1_filename = next(iter(uploaded))
c1_output = json.loads(uploaded[c1_filename].decode('utf-8'))
primary = c1_output['matched_ground_truths'][0]
instructions = primary['radiologist_instructions']
modalities = instructions.get('suggested_modalities', [])
modality = modalities[0] if modalities else ''

# El modelo oficial solo condiciona anatomía/modalidad. Este mapeo no transfiere la lesión del prompt.
normalized = modality.lower()
if 'mri' in normalized or 'mrcp' in normalized:
    if 'abdomen' not in normalized and 'mrcp' not in normalized:
        raise ValueError(f'MRI no cubierta por este notebook: {modality}. Elegí una clase oficial compatible.')
    CLASS_ID = 5
elif 'cervical' in normalized or 'head' in normalized or 'neck' in normalized:
    CLASS_ID = 0
else:
    CLASS_ID = 1  # CT de tórax o abdomen

LATENT_RESOLUTION = (16, 32, 32)
SEED = 20260718
print(f'GT C1: {primary["gt_id"]} | modalidad: {modality} | clase MedDiffusion: {CLASS_ID}')
print('Prompt clínico conservado para metadatos:', instructions.get('meddiffusion_reference_prompt', ''))
from pathlib import Path

def find_checkpoint(filename):
    matches = list(Path('checkpoints').rglob(filename))
    if not matches:
        raise FileNotFoundError(f'No se encontró {filename}. Revisá la descarga de checkpoints.')
    return str(matches[0])

AE_CKPT = find_checkpoint('PatchVolume_8x_s2.ckpt')
MODEL_CKPT = find_checkpoint('BiFlowNet_0453500.pt')
OUTPUT_DIR = 'oncobridge_meddiffusion_output'
print(AE_CKPT, MODEL_CKPT)

## 4. Generar un único volumen

El script oficial genera todas las clases. Esta versión conserva su arquitectura e inferencia, pero genera solo la anatomía elegida para que el notebook sea útil para C2.

In [ ]:
%%writefile generate_one_volume.py
import argparse, os, random
import numpy as np
import torch
import torchio as tio
from ddpm.BiFlowNet import GaussianDiffusion
from ddpm import BiFlowNet
from AutoEncoder.model.PatchVolume import patchvolumeAE

SPACING = {0: (1, 1, 1), 1: (1.25, 1.25, 1.25), 5: (1.2, 1.2, 2)}
NAMES = {0: 'CTHeadNeck', 1: 'CTChestAbdomen', 5: 'MRAbdomen'}

def main(args):
    if not torch.cuda.is_available():
        raise RuntimeError('Se requiere GPU CUDA para 3D MedDiffusion.')
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    device = torch.device('cuda')
    model = BiFlowNet(dim=72, dim_mults=[1, 1, 2, 4, 8], channels=8, init_kernel_size=3,
        cond_classes=7, learn_sigma=False, use_sparse_linear_attn=[0, 0, 0, 1, 1],
        vq_size=64, num_mid_DiT=1, patch_size=1).to(device)
    diffusion = GaussianDiffusion(channels=8, timesteps=1000, loss_type='l1').to(device)
    checkpoint = torch.load(args.model_ckpt, map_location='cpu')
    model.load_state_dict(checkpoint['ema'], strict=True)
    model.eval()
    autoencoder = patchvolumeAE.load_from_checkpoint(args.ae_ckpt).to(device).eval()
    res = tuple(args.latent_resolution)
    with torch.no_grad():
        z = torch.randn(1, 8, *res, device=device)
        y = torch.tensor([args.class_id], device=device)
        res_emb = torch.tensor(res, device=device) / 64.0
        latent = diffusion.sample(model, z, y=y, res=res_emb, strategy='ddpm')
        latent = (((latent + 1.0) / 2.0) * (autoencoder.codebook.embeddings.max() - autoencoder.codebook.embeddings.min())) + autoencoder.codebook.embeddings.min()
        volume = autoencoder.decode(latent, quantize=True)
    volume = volume.detach().squeeze(0).cpu().transpose(1, 3).transpose(1, 2)
    os.makedirs(args.output_dir, exist_ok=True)
    name = NAMES[args.class_id]
    path = os.path.join(args.output_dir, f'{name}_seed_{args.seed}.nii.gz')
    tio.ScalarImage(tensor=volume, affine=np.diag(SPACING[args.class_id] + (1,))).save(path)
    print(path)

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--ae-ckpt', required=True)
    parser.add_argument('--model-ckpt', required=True)
    parser.add_argument('--output-dir', required=True)
    parser.add_argument('--class-id', type=int, choices=[0, 1, 5], required=True)
    parser.add_argument('--latent-resolution', nargs=3, type=int, required=True)
    parser.add_argument('--seed', type=int, default=20260718)
    main(parser.parse_args())


In [ ]:
latent_arg = ' '.join(map(str, LATENT_RESOLUTION))
!python generate_one_volume.py --ae-ckpt "{AE_CKPT}" --model-ckpt "{MODEL_CKPT}" --output-dir "{OUTPUT_DIR}" --class-id {CLASS_ID} --latent-resolution {latent_arg} --seed {SEED}

## 5. Exportar un corte PNG para C2

C2 recibe PNG/JPG/WEBP. La celda elige un corte central y normaliza intensidades con percentiles solo para visualizarlo.

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np
from PIL import Image
from google.colab import files

volume_path = next(Path(OUTPUT_DIR).glob('*.nii.gz'))
volume = np.squeeze(nib.load(volume_path).get_fdata())
slice_2d = volume[:, :, volume.shape[2] // 2]
low, high = np.percentile(slice_2d, (1, 99))
rendered = np.clip((slice_2d - low) / max(high - low, 1e-8), 0, 1)
png_path = Path(OUTPUT_DIR) / 'meddiffusion_reference_slice.png'
Image.fromarray((rendered * 255).astype(np.uint8)).save(png_path)
display(Image.open(png_path))
files.download(str(png_path))

## 6. Usar la referencia en OncoBridge C2

Descargá `meddiffusion_reference_slice.png` y cargalo en la sección **Componente 2** de Streamlit como referencia sintética. También se puede pasar por consola:

```powershell
python onco_bridge_c1\run_component2.py onco_bridge_c1\artifacts\c1_case_001.json --device cuda --output-dir generated_references\case_001_local
```

El estudio real del paciente sigue siendo obligatorio para que C2 haga un análisis visual. La imagen creada aquí solo aporta una referencia anatómica sintética.